# Exploración de estudios de Movebank

Este notebook consulta la API **real** de Movebank, obtiene los estudios accesibles y muestra los que tienen **más de tres individuos**. Después permite consultar individuos y eventos GPS de un estudio elegido.

La aplicación del proyecto usa el mismo cliente para descargar estudios, individuos y posiciones; guarda una caché local SQLite y presenta métricas, gráficos y mapas en Streamlit. Las consultas de este notebook se mantienen en memoria y **no actualizan la caché del dashboard**.

## 1. Preparación y credenciales

En VS Code, selecciona el kernel Python de `movebank_migracion\.venv` y ejecuta las celdas en orden. Si falta el soporte Jupyter, desde la carpeta del proyecto ejecuta:

```powershell
.\.venv\Scripts\python.exe -m pip install -e ".[notebook]"
```

Configura `MOVEBANK_USERNAME` y `MOVEBANK_PASSWORD` en `.env` dentro de la carpeta del proyecto, o como variables de entorno. Las variables de entorno tienen prioridad. El archivo está excluido de Git, pero no está cifrado: no lo compartas ni imprimas tus credenciales. Sin credenciales se intenta el acceso anónimo; el servidor puede exigir autenticación.

Este notebook cambia el directorio de trabajo a la carpeta del proyecto para que el cliente encuentre `.env`. Inícialo desde la carpeta del proyecto o desde la raíz de este repositorio.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

actual = Path.cwd()
proyecto = actual if (actual / 'consulta_movebank.ipynb').is_file() else actual / 'movebank_migracion'
if not (proyecto / 'consulta_movebank.ipynb').is_file():
    raise FileNotFoundError('Inicia el kernel desde la raíz del repositorio o desde movebank_migracion.')
os.chdir(proyecto)

from movebank.application.cli import with_consent
from movebank.infrastructure.config import credentials
from movebank.infrastructure.movebank.client import MovebankClient, URL
from movebank.infrastructure.movebank.services import (
    MovebankEventService,
    MovebankIndividualService,
    MovebankStudyService,
)

print('Proyecto preparado. Las credenciales no se muestran.')

## 2. Cómo se consulta la API

El cliente realiza una petición HTTP **GET** a `https://www.movebank.org/movebank/service/direct-read`, con autenticación HTTP Basic cuando hay credenciales. Para el catálogo utiliza `entity_type=study` e `i_have_download_access=true`. Este filtro incluye estudios públicos y, si hay una cuenta autenticada, otros estudios a los que dicha cuenta tenga acceso.

Este endpoint devuelve **CSV**, no JSON. El cliente lo interpreta como una lista de diccionarios; después pandas permite filtrarlo. Movebank también ofrece [endpoints JSON para ubicaciones](https://github.com/movebank/movebank-api-doc/blob/master/movebank-api.md#accessing-the-database-using-jsonjavascript-requests), distintos de esta consulta del catálogo.

El cliente reutiliza la sesión HTTP, limita la frecuencia dentro del proceso, reintenta errores transitorios y comunica los errores. Movebank limita a una solicitud simultánea por IP: no ejecutes consultas al mismo tiempo desde el dashboard y este notebook.

In [ ]:
print('Endpoint:', URL)
print('Parámetros GET del catálogo:', {'entity_type': 'study', 'i_have_download_access': 'true'})

cliente = MovebankClient(*credentials())
try:
    registros = MovebankStudyService(cliente).list()
finally:
    cliente.close()

print(f'Estudios accesibles recibidos: {len(registros)}')

## 3. Estudios con más de tres individuos

`number_of_individuals` es un resumen del estudio proporcionado por Movebank, actualizado aproximadamente una vez al día. No garantiza que todos los individuos tengan posiciones GPS disponibles ni que todos los registros sean descargables. Los recuentos desconocidos se excluyen y se informan por separado. El filtro es estrictamente `> 3`.

In [ ]:
if not registros:
    raise RuntimeError('La consulta no devolvió estudios accesibles para esta cuenta.')

estudios = pd.DataFrame(registros)
columnas = ['id', 'name', 'number_of_individuals', 'number_of_deployed_locations', 'taxon_ids', 'sensor_type_ids']
faltantes = set(columnas) - set(estudios.columns)
if faltantes:
    raise ValueError(f'La respuesta de Movebank no incluye las columnas esperadas: {sorted(faltantes)}')

estudios['number_of_individuals'] = pd.to_numeric(estudios['number_of_individuals'], errors='coerce')
sin_recuento = int(estudios['number_of_individuals'].isna().sum())
estudios_filtrados = estudios.loc[estudios['number_of_individuals'] > 3, columnas].sort_values(
    ['number_of_individuals', 'name'], ascending=[False, True]
).reset_index(drop=True)
print(f'Estudios con más de 3 individuos: {len(estudios_filtrados)}')
print(f'Estudios sin recuento numérico informado: {sin_recuento}')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(estudios_filtrados)

## 4. Representación JSON de los resultados

Para inspeccionar la estructura, esta celda convierte los tres primeros resultados filtrados a JSON. Es una **conversión local del CSV**, no la respuesta original del endpoint JSON de Movebank.

In [ ]:
print(estudios_filtrados.head(3).to_json(orient='records', force_ascii=False, indent=2))

## 5. Consultar individuos y eventos GPS (opcional)

Elige un `id` de la tabla anterior y asígnalo a `ESTUDIO_ID`. Si también quieres descargar eventos GPS, asigna un identificador interno de la tabla de individuos a `INDIVIDUO_ID`. Con `None` no se realiza esa descarga. El identificador interno puede cambiar en Movebank; conserva también el identificador local dentro del estudio.

Se realizan GET con `entity_type=individual` o `entity_type=event`, `study_id` y, para eventos, `individual_id` y `sensor_type_id=653` (GPS). La consulta de eventos puede ser grande. Si Movebank solicita aceptar una licencia, lee el texto mostrado y escribe `ACEPTO` solo si estás de acuerdo; la misma sesión envía el MD5 requerido. También se aplican los [términos generales](https://www.movebank.org/cms/movebank-content/general-movebank-terms-of-use) y la [política de datos](https://www.movebank.org/cms/movebank-content/data-policy).

In [ ]:
ESTUDIO_ID = None  # Sustituye por un id de la tabla para consultar individuos.
INDIVIDUO_ID = None  # Sustituye por un id de individuo para consultar GPS.

if ESTUDIO_ID is None:
    print('Consulta opcional omitida: selecciona ESTUDIO_ID para ver individuos.')
else:
    cliente = MovebankClient(*credentials())
    try:
        individuos = with_consent(lambda md5: MovebankIndividualService(cliente).list(ESTUDIO_ID, md5))
        print(f'Individuos recibidos: {len(individuos)}')
        display(pd.DataFrame(individuos))
        if INDIVIDUO_ID is not None:
            eventos = with_consent(
                lambda md5: MovebankEventService(cliente).list(ESTUDIO_ID, INDIVIDUO_ID, md5)
            )
            print(f'Eventos GPS recibidos: {len(eventos)}')
            display(pd.DataFrame(eventos).head(20))
    finally:
        cliente.close()

## 6. Interpretación y uso

- En el dashboard, selecciona un estudio, actualiza sus individuos y descarga los GPS de uno de ellos para generar las vistas. Las consultas del notebook y la caché del dashboard son independientes.
- Los datos pueden incluir posiciones sin coordenadas o marcadas como `visible=false`; revisa la calidad y las licencias antes de analizar o redistribuir información. Las métricas son estimaciones, no una reconstrucción continua exacta.
- El notebook puede mostrar información privada accesible a tu cuenta. Borra las salidas antes de compartirlo si contienen datos restringidos.
- Consulta la [documentación oficial de la API](https://github.com/movebank/movebank-api-doc/blob/master/movebank-api.md) y cita cada estudio conforme a sus condiciones.